# Week 8 held-out classification confirmation: Qwen2.5-1.5B

This notebook runs the held-out Week 8 classification matrix on Kaggle T4 x2: SST-2, QNLI, MNLI matched and MNLI mismatched, with seeds 6101-6106 under the non-IID high-staleness regime.

`RUN_MODE='extended'` runs only the five new federated-LoRA baselines: 4 tasks x 5 methods x 6 seeds = 120 jobs. `RUN_MODE='core'` is the exact Week 8 set (192 jobs), and `RUN_MODE='combined'` runs all 13 methods (312 jobs). Existing valid results are skipped by the Week 8 runner; attached output roots can be imported with `RESUME_ROOTS`.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile

REPO_URL = 'https://github.com/TrgPhan/VASTLoRA.git'
REPO_REF = '8b347cab1a553708c3616f429a9da4bedd2c54a6'
RUN_MODE = 'extended'  # smoke / core / extended / combined
RUN_TRAINING = False
SHARD_COUNT = 1
SHARD_INDEX = 0
GPU_IDS = [0, 1]
FORCE_RERUN = False
RESUME_ROOTS = []  # e.g. ['/kaggle/input/week8-old-output/week8_core']
CACHE_ASSETS = True
RUN_ANALYSIS = False
WORK_ROOT = Path('/kaggle/working')
REPO_DIR = WORK_ROOT / 'VASTLoRA-week8-classification'
if RUN_MODE not in {'smoke', 'core', 'extended', 'combined'}:
    raise ValueError('RUN_MODE must be smoke, core, extended or combined')
if SHARD_COUNT < 1 or not 0 <= SHARD_INDEX < SHARD_COUNT:
    raise ValueError('invalid shard selection')
if RUN_MODE == 'smoke' and RUN_TRAINING:
    print('Smoke is diagnostic only; use core or extended for confirmation jobs.')
WORK_ROOT.mkdir(parents=True, exist_ok=True)
print({'mode': RUN_MODE, 'training': RUN_TRAINING, 'repo_ref': REPO_REF, 'shard': [SHARD_INDEX, SHARD_COUNT]})

In [ ]:
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', 'checkout', '--detach', REPO_REF], cwd=REPO_DIR, check=True)
resolved_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
dirty = subprocess.check_output(['git', 'status', '--porcelain', '--untracked-files=no'], cwd=REPO_DIR, text=True).strip()
if resolved_commit != REPO_REF or dirty:
    raise RuntimeError('The checkout is not the pinned clean release.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[scale,dev]'], cwd=REPO_DIR, check=True)
RUNNER = REPO_DIR / 'scripts/run_week8_classification_matrix.py'
MATRIX_SOURCE = REPO_DIR / 'configs/rift_core_heldout_confirmation_matrix.json'
if not RUNNER.exists() or not MATRIX_SOURCE.exists():
    raise RuntimeError('Week 8 runner or held-out matrix is missing.')
print({'commit': resolved_commit, 'repo': str(REPO_DIR)})

In [ ]:
matrix = json.loads(MATRIX_SOURCE.read_text())
core_methods = ['raw', 'freshness', 'fedrot', 'spectral_surgery', 'alignfed_calibration', 'rift', 'rift_diag', 'rift_core']
extended_methods = ['fedavg_lora', 'fedex_lora', 'flora_lora', 'ffa_lora', 'florg']
combined_methods = core_methods + extended_methods
if RUN_MODE == 'smoke':
    selected_tasks = ['sst2']
    selected_methods = ['raw', 'freshness', 'rift_core']
    selected_seeds = [6101]
else:
    selected_tasks = [task['name'] for task in matrix['tasks']]
    if RUN_MODE == 'core':
        selected_methods = core_methods
    elif RUN_MODE == 'extended':
        selected_methods = extended_methods
    else:
        selected_methods = combined_methods
    selected_seeds = [int(seed) for seed in matrix['seeds']]
selected_regimes = ['noniid_high_staleness']
matrix['methods'] = selected_methods
matrix['seeds'] = selected_seeds
matrix['regimes'] = [r for r in matrix['regimes'] if r['name'] in selected_regimes]
matrix['tasks'] = [t for t in matrix['tasks'] if t['name'] in selected_tasks]
MATRIX = WORK_ROOT / f'week8_{RUN_MODE}_heldout_matrix.json'
MATRIX.write_text(json.dumps(matrix, indent=2), encoding='utf-8')
OUTPUT_ROOT = WORK_ROOT / f'week8_qwen15b_{RUN_MODE}_heldout'
task_by_name = {task['name']: task for task in matrix['tasks']}
all_jobs = [(task['name'], 'noniid_high_staleness', method, seed)
            for task in matrix['tasks'] for method in selected_methods for seed in selected_seeds]
jobs = all_jobs[SHARD_INDEX::SHARD_COUNT]
print({'tasks': selected_tasks, 'regimes': selected_regimes, 'methods': selected_methods,
       'seeds': selected_seeds, 'total_jobs': len(all_jobs), 'shard_jobs': len(jobs),
       'output_root': str(OUTPUT_ROOT)})

In [ ]:
# Optional asset cache. It does not change the experiment or data split.
if CACHE_ASSETS:
    from huggingface_hub import snapshot_download
    from datasets import load_dataset
    model_cfg = json.loads((REPO_DIR / 'configs/local_1_5b_rift_development.json').read_text())['model']
    snapshot_download(model_cfg['name'], revision=model_cfg['revision'])
    for subset in ('sst2', 'qnli', 'mnli'):
        load_dataset('nyu-mll/glue', subset)
    print('Cached Qwen2.5-1.5B and GLUE assets.')

# Validate every selected job without training.
dry = [sys.executable, str(RUNNER), '--matrix', str(MATRIX), '--output-root', str(OUTPUT_ROOT), '--dry-run']
subprocess.run(dry, cwd=REPO_DIR, check=True)
print('Preflight passed.')

In [ ]:
# Import old output roots before launching. The runner still validates matrix, config and git identity,
# so stale or incompatible result.json files are rerun rather than treated as evidence.
for source_root in RESUME_ROOTS:
    source_root = Path(source_root)
    if not source_root.exists():
        raise FileNotFoundError(source_root)
    for result in source_root.rglob('result.json'):
        relative = result.relative_to(source_root)
        destination = OUTPUT_ROOT / relative
        if not destination.exists():
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copytree(result.parent, destination.parent, dirs_exist_ok=True)
print('Resume roots imported:', RESUME_ROOTS)

if RUN_TRAINING:
    import torch
    if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
        raise RuntimeError(f'Expected T4 x2, found {torch.cuda.device_count()} CUDA device(s).')
    if len(GPU_IDS) != 2 or len(set(GPU_IDS)) != 2:
        raise ValueError('GPU_IDS must contain two distinct GPU ids.')
    log_dir = OUTPUT_ROOT / 'kaggle_logs'
    log_dir.mkdir(parents=True, exist_ok=True)
    failures = []
    for offset in range(0, len(jobs), len(GPU_IDS)):
        wave = jobs[offset:offset + len(GPU_IDS)]
        active = []
        for index, (task, regime, method, seed) in enumerate(wave):
            task_spec = task_by_name[task]
            command = [sys.executable, '-u', str(RUNNER), '--matrix', str(MATRIX), '--output-root', str(OUTPUT_ROOT),
                       '--task', task, '--regime', regime, '--method', method, '--seed', str(seed),
                       '--eval-offset', str(task_spec.get('eval_offset', 0))]
            if FORCE_RERUN:
                command.append('--force')
            log_path = log_dir / f'{task}_{method}_seed{seed}_gpu{GPU_IDS[index]}.log'
            handle = log_path.open('w', encoding='utf-8')
            env = os.environ.copy()
            env['CUDA_VISIBLE_DEVICES'] = str(GPU_IDS[index])
            env['TOKENIZERS_PARALLELISM'] = 'false'
            env['PYTHONUNBUFFERED'] = '1'
            env['OMP_NUM_THREADS'] = '1'
            env['MKL_NUM_THREADS'] = '1'
            env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
            process = subprocess.Popen(command, cwd=REPO_DIR, env=env, stdout=handle, stderr=subprocess.STDOUT)
            active.append((process, handle, (task, regime, method, seed), log_path))
            print('started', task, method, seed, 'GPU', GPU_IDS[index])
        while any(process.poll() is None for process, _, _, _ in active):
            time.sleep(5)
        for process, handle, job, log_path in active:
            handle.close()
            print({'job': job, 'return_code': process.returncode, 'log': str(log_path)})
            if process.returncode != 0:
                failures.append((job, str(log_path)))
    if failures:
        raise RuntimeError(f'{len(failures)} jobs failed; inspect logs: {failures}')
    print('Completed shard jobs:', len(jobs))
else:
    print('Preflight only: set RUN_TRAINING=True after reviewing the job list.')

In [ ]:
if RUN_ANALYSIS:
    analyzer = REPO_DIR / 'scripts/analyze_kaggle_3b_rift_competitors.py'
    analysis_dir = OUTPUT_ROOT / 'analysis'
    subprocess.run([sys.executable, str(analyzer), '--input-dir', str(OUTPUT_ROOT),
                    '--output-dir', str(analysis_dir), '--matrix', str(MATRIX)],
                   cwd=REPO_DIR, check=True)
    report = analysis_dir / 'week8_verdict.md'
    if report.exists():
        print(report.read_text(encoding='utf-8'))

archive = WORK_ROOT / f'week8_qwen15b_{RUN_MODE}_heldout_results.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    if OUTPUT_ROOT.exists():
        for path in sorted(OUTPUT_ROOT.rglob('*')):
            if path.is_file() and path.name != '.launcher.lock':
                z.write(path, path.relative_to(OUTPUT_ROOT.parent))
print('Archive:', archive)